# E3 · Permutation on the training set rewards the noise the forest memorized

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Find out why a report "corrected with the permutation" puts `ticket_code` (a random integer) ahead of `age` and `siblings_spouses`: the permutation was computed on the **training set**, where the forest has memorized the noise too. Redo the computation on the test set and measure the difference.

## Dataset

`ds.titanic()` with two useless columns added (`random_number`, `ticket_code`); 70/30 split. Pipeline: median, mode + `OrdinalEncoder`, `RandomForestClassifier(300, random_state=0)`; `pipe` is already trained. Scoring: AUC.

## Educational purpose

Remember that **every importance measure must be evaluated on data the model has not seen**. A forest has AUC 1.0 on the training set: shuffling any column, even a noise one, makes it get worse because it has learned to use it to recognize the single examples.

## Task

1. Run the colleague's cell: `train_perm` (`permutation_importance` computed on `X_tr, y_tr`, `n_repeats=10`, `random_state=0`, scoring `roc_auc`, as a `pd.Series` indexed by the columns) and `train_top5`.
2. Compute `test_perm` (same computation on `X_te, y_te`) and `test_top5`. Save `train_auc` and `test_auc` of the model: the difference explains the phenomenon.
3. Save `train_noise` and `test_noise` (sum of the positive importances of `random_number` and `ticket_code` for the two computations) and `train_code_rank` / `test_code_rank` (position, from 1, of `ticket_code` in the descending order).
4. **Stability:** compute the importances on the test set on 3 random subsamples of half of the test set (`idx = np.random.default_rng(s).choice(len(X_te), len(X_te) // 2, replace=False)`, `s = 0, 1, 2`, `n_repeats=5`) and save `stable` (True if `ticket_code` has an importance below 0.02 in all three).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

df = ds.titanic()
X, y = df.drop(columns="survived").copy(), df["survived"]
r = np.random.default_rng(0)
X["random_number"] = r.uniform(size=len(X))
X["ticket_code"] = r.integers(0, 100000, len(X))
NUM = ["pclass", "age", "siblings_spouses", "parents_children", "fare", "random_number", "ticket_code"]
CAT = ["sex", "port", "deck", "title"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM), ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder()), CAT)])
pipe = make_pipeline(prep, RandomForestClassifier(300, random_state=0, n_jobs=-1)).fit(X_tr, y_tr)
NOISE = ["random_number", "ticket_code"]


def importances(A, b, n_repeats=10, seed=0):
    r = permutation_importance(pipe, A, b, n_repeats=n_repeats, random_state=seed, scoring="roc_auc")
    return pd.Series(r.importances_mean, index=A.columns)

## Your work

### The colleague's code

In [ ]:
train_perm = importances(X_tr, y_tr)
train_top5 = list(train_perm.nlargest(5).index)
print("top 5 by permutation (computed on the training set):", train_top5)
print(train_perm.round(3).sort_values(ascending=False).head(7).to_string())

### Your diagnosis and the fix

In [ ]:
test_perm = None
test_top5 = None
train_auc = None
test_auc = None

In [ ]:
train_noise = None
test_noise = None
train_code_rank = None
test_code_rank = None

In [ ]:
stable = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`test_perm = importances(X_te, y_te)`. AUC: `roc_auc_score(y_tr, pipe.predict_proba(X_tr)[:, 1])`. Rank: `perm.rank(ascending=False)["ticket_code"]`. Subsamples: `idx = np.random.default_rng(s).choice(len(X_te), len(X_te) // 2, replace=False)`; `importances(X_te.iloc[idx], y_te.iloc[idx], 5)`.'):
    check("ticket_code" in train_top5, "the colleague's report puts `ticket_code` in the top five", "don't modify the colleague's code")
    check(train_auc is not None and train_auc > 0.99 and test_auc < 0.9, "training AUC 1.0 (memorization); test below 0.9", "pipe.score(...) is not the AUC: use roc_auc_score with predict_proba")
    check(test_perm is not None and "ticket_code" not in test_top5 and "random_number" not in test_top5 and test_top5[0] == "sex", "on the test set the two noise columns are not in the top five and `sex` is first", "importances(X_te, y_te)")
    check(train_noise > 0.03 and test_noise < 0.02 and train_code_rank < test_code_rank, "the noise is worth over 0.03 on the training set and less than 0.02 on the test set; the ticket code drops in the ranking", "sum of the positive importances of the two noise columns")
    check(stable is True, "on three subsamples of the test set `ticket_code` always stays below 0.02", "n_repeats=5, random subsamples of the test set")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_permutation_importance_computed_on_the_training_set.ipynb`. Open it after you pass the check, or after you have really tried.